# Outlier detection methods (Catalog 2)

Runs a set of unsupervised, non-parametric outlier-detection methods on the cleaned
Catalog 2 features, each writing a per-burst score and rank. They carry different inductive
biases so that, between them, they cover the standard anomaly geometries:

- **Local Outlier Factor (LOF)**, density relative to local neighbours, swept over k (local).
- **Extended Isolation Forest (EIF)**, isolation by random hyperplane splits (point).
- **kNN distance**, absolute mean distance to the k nearest neighbours, swept over k
  (point/local).
- **CBLOF**, cluster-based, targeting small off-population clusters (collective).
- **Random-subspace ensemble**, feature bagging for anomalies hiding in a feature
  combination (subspace).
- **Conditional anomaly detection (CAD)**, morphology atypical for a burst's observational
  context, swept over k (contextual), non-parametric.

LOF, kNN and CAD all depend on a neighbourhood-size parameter with no single correct
value, so all three are swept over the same k bracket and summarised by median rank and
stability, rather than scrutinising LOF for k-sensitivity while leaving the other two
fixed at one unexamined value.

Scoring is label-blind: the repeater flag is not used. Three methods were trialled and
removed as ill-suited to this heavy-tailed, multimodal data: a Gaussian-mixture density
method (no BIC minimum; component count unpinnable, candidate set unstable), its GMM-based
contextual variant, and robust Mahalanobis (MCD) (assumes a single elliptical core, which
the data violates; its correlation-aware role is taken over by the random-subspace ensemble
without that assumption). CAD is reinstated here in a non-parametric form. Method selection
and per-geometry sensitivity are assessed by injection-recovery (see the validation redesign).

All six scorers are implemented once, in `src/frb_anomaly/methods.py`, and imported here
rather than reimplemented. The injection-recovery and candidate-validation notebooks score
through the same functions, so a method's behaviour is defined in exactly one place.

## Setup: load the cleaned features

In [1]:
# Make the shared code in src/ importable, same two-hop pattern as the cleaning notebook.
import sys
from pathlib import Path

PROJECT = Path.cwd().parent.parent.parent
sys.path.insert(0, str(PROJECT / 'src'))

import numpy as np
import pandas as pd

# All six scorers are implemented once in frb_anomaly.methods (the canonical, unit-tested
# form) and imported rather than reimplemented, so this notebook, the injection-recovery
# validation and the candidate-validation notebook always score against identical code.
from frb_anomaly.methods import (
    lof_scores,
    eif_scores,
    knn_scores,
    cblof_scores,
    subspace_scores,
    cad_scores,
)

# Load the scaled features written by 01_cleaning_cat2.ipynb.
scaled_path = PROJECT / 'data' / 'processed' / 'phase_1' / 'catalog2_features_scaled.csv'
scaled = pd.read_csv(scaled_path)

# is_repeater round-trips through CSV as the string 'True'/'False', not a bool. Cast it
# back explicitly so later code can use it as a boolean mask.
scaled['is_repeater'] = scaled['is_repeater'].astype(str).str.lower() == 'true'

# Split ID/label columns from the feature columns. X is what the methods see.
id_cols = ['tns_name', 'sub_num', 'is_repeater', 'repeater_name', 'catalog1_flag']
feature_cols = [c for c in scaled.columns if c not in id_cols]
X = scaled[feature_cols].values

print(f'Loaded {len(scaled)} sub-burst rows, {len(feature_cols)} features')
print(f'  repeaters: {int(scaled["is_repeater"].sum())}')
print(f'  one-offs:  {int((~scaled["is_repeater"]).sum())}')
print(f'  features:  {feature_cols}')

Loaded 4109 sub-burst rows, 8 features
  repeaters: 1047
  one-offs:  3062
  features:  ['dm_fitb', 'width_fitb', 'flux', 'fluence', 'sp_idx', 'sp_run', 'peak_freq', 'bandwidth']


## Method 1: Local Outlier Factor (LOF), swept over k

**Principle: local density.** Where Mahalanobis asks 'how far is this burst from the
global centre?', LOF asks a more local question: 'is this burst sitting in a sparser
patch than its own neighbours are?' It compares each burst's local density to the density
around its k nearest neighbours. A burst packed in among similar-density neighbours
scores around 1 (an inlier); a burst in a relatively empty pocket compared to its
neighbours scores well above 1 (an outlier). This catches a different kind of anomaly: a
burst can be near the global centre and still be locally isolated, which Mahalanobis
would miss.

**Why sweep k.** LOF's entire answer depends on what counts as 'local', set by k, the
neighbourhood size. Too small and the score reacts to noise; too large and 'local' stops
being local. There is no single correct k, so run a range and look at how stable each
burst's ranking is across the sweep. Cat 1 used {10,20,30,50,75} around sqrt(558)~24.
Cat 2 has ~8x more rows, so re-bracket around sqrt(N)~67 with {30,50,75,100,150}, so
'local' covers the same fraction of the data it did for Cat 1.

**Reading the sweep.** A burst that stays near the top for every k is a robust local
outlier one can trust. A burst extreme at one k and ordinary at the others is a k-artefact,
not a real anomaly. Summarise each burst by its median rank across the five k
values (its consistent standing) and separately count, as a stability score out of
5, how many of the sweeps put it in the top 10%.

In [2]:
# LOF is now the first scored method, so it seeds the results table with the IDs and
# label; the later methods add their own score/rank columns.
results = scaled[['tns_name', 'sub_num', 'is_repeater']].copy()

# The k values to sweep, reused below for kNN and CAD so all three neighbourhood-size
# sweeps are directly comparable, and the size of the 'top tier' used for the stability
# count.
K_VALUES = [30, 50, 75, 100, 150]
N = len(scaled)
top_n = int(round(0.10 * N))  # top 10% of rows

# For each k: score with lof_scores (already the shared 'bigger = more outlying'
# convention) and rank. Collect one rank column per k so one can watch each burst move
# across the sweep.
lof_ranks = pd.DataFrame(index=scaled.index)
for k in K_VALUES:
    score_k = lof_scores(X, k=k)
    lof_ranks[k] = pd.Series(score_k, index=scaled.index).rank(ascending=False, method='first').astype(int)

# Summarise the sweep by the MEDIAN rank across the five k values. Median, not mean, so a
# single stray k cannot drag a burst up or down. Lower median = consistently outlying.
results['lof_median_rank'] = lof_ranks.median(axis=1)
# Re-rank the median into a clean 1..N integer alongside the other method ranks.
results['lof_rank'] = results['lof_median_rank'].rank(ascending=True, method='first').astype(int)
# Stability: in how many of the 5 sweeps did this burst land in the top 10%?
results['lof_stability'] = (lof_ranks <= top_n).sum(axis=1)

# Show the 10 most outlying rows by median rank, with their rank at EACH k so the
# stability (or instability) across the sweep is visible directly.
top = results.sort_values('lof_rank').head(10)
detail = lof_ranks.loc[top.index].copy()
detail.columns = [f'k={k}' for k in K_VALUES]
show = pd.concat([scaled.loc[top.index, ['tns_name', 'sub_num', 'is_repeater']], detail,
                  results.loc[top.index, ['lof_stability']]], axis=1)
print('Top 10 by LOF median rank, with rank at each k:')
print(show.to_string(index=False))
print(f'\nRepeaters in this top 10: {int(scaled.loc[top.index, "is_repeater"].sum())} of 10')
print(f'(top tier = top {top_n} rows = top 10%; stability is out of {len(K_VALUES)} sweeps)')

Top 10 by LOF median rank, with rank at each k:
    tns_name  sub_num  is_repeater  k=30  k=50  k=75  k=100  k=150  lof_stability
FRB20220407A        0        False     1     1     1      1      1              5
FRB20200122B        1         True     3     2     2      4      5              5
FRB20221219D        0         True     2     3     3      5      8              5
FRB20200705D        0        False    40     9     4      2      3              5
FRB20210512C        0        False    65    21     5      6      4              5
FRB20210222C        0        False    96    23     6      7      6              5
FRB20211108A        1        False    32    14     7      3      2              5
FRB20190929C        1         True    12     5     8      9     11              5
FRB20200429A        0        False    99    31     9      8      7              5
FRB20230826A        1         True     7     6    10     11     14              5

Repeaters in this top 10: 4 of 10
(top tier = top

## Method 2: Extended Isolation Forest (EIF)

**Principle: ease of isolation.** This asks yet another question: 'how few random cuts
does it take to fence this burst off on its own?' It builds many random binary trees,
each repeatedly slicing the data into smaller regions. A normal burst sits deep in the
crowd and needs many cuts before it ends up alone; an anomaly, being rare and different,
gets isolated in a handful of cuts. Average that path length over a whole forest and you
have an anomaly score: short path = easily isolated = outlier. This needs no notion of
distance or density at all, which is why it can catch things the other two miss.

**Why the Extended version.** Standard Isolation Forest (scikit-learn's) only ever cuts
straight across one feature at a time; its splits are axis-aligned. That leaves a known
blind spot: artefact bands of artificially low score running parallel to the axes, and it
struggles with anomalies that are only unusual in a diagonal combination of features.
Extended Isolation Forest (Hariri et al. 2019, arXiv:1811.02141) replaces the
axis-aligned cuts with **random hyperplanes**, slicing at random angles through several
features at once, removing that bias while keeping the isolation principle untouched.
This is why `isotree` is used rather than scikit-learn.

**Settings.** Full extension (`ndim` set to all 8 features, so every cut is a hyperplane
through the whole feature space), 100 trees, and a 256-burst subsample per tree (the
classic isolation-forest sampling from Liu et al. 2008). A fixed seed keeps the run
reproducible. The score is already standardised to [0, 1], higher = more easily isolated
= more outlying, matching the other two methods.

In [3]:
# Settings (full extension, 100 trees, 256-burst subsample, fixed seed) are documented
# and fixed inside eif_scores; see the settings paragraph above for the rationale.
eif_score = eif_scores(X)

results['eif_score'] = eif_score
# rank 1 = most outlying, the same convention as the other methods.
results['eif_rank'] = results['eif_score'].rank(ascending=False, method='first').astype(int)

# Top 10 by this method, with the repeater flag, same preview as the other two.
top10 = results.sort_values('eif_rank').head(10).copy()
top10['eif_score'] = top10['eif_score'].round(3)
print('Top 10 by Extended Isolation Forest anomaly score:')
print(top10[['eif_rank', 'tns_name', 'sub_num', 'eif_score', 'is_repeater']].to_string(index=False))
print(f'\nRepeaters in this top 10: {int(top10["is_repeater"].sum())} of 10')

/Users/elicox/opt/anaconda3/envs/FRB_Research/lib/python3.11/site-packages/isotree/__init__.py:97: UserWarning: Attempting to use more than 1 thread, but package was built without multi-threading support - see the project's GitHub page for more information.
  warnings.warn(msg_omp)


Top 10 by Extended Isolation Forest anomaly score:
 eif_rank     tns_name  sub_num  eif_score  is_repeater
        1 FRB20220407A        0      0.678        False
        2 FRB20190202A        0      0.646        False
        3 FRB20200723B        0      0.640        False
        4 FRB20210909B        0      0.640        False
        5 FRB20211108A        1      0.634        False
        6 FRB20210427A        0      0.633        False
        7 FRB20230911A        0      0.628        False
        8 FRB20201015B        0      0.624        False
        9 FRB20230314A        0      0.622        False
       10 FRB20220222B        0      0.618        False

Repeaters in this top 10: 0 of 10


## Method 3: kNN distance, swept over k

**Principle: absolute distance to neighbours.** For each burst, the mean Euclidean
distance to its k nearest neighbours measures how isolated it is in absolute terms. This
is the distance-based counterpart to LOF: where LOF compares a burst's density to its
neighbours' (a ratio, flagging locally sparse points even near the centre), kNN uses
absolute distance, so it responds to globally isolated bursts. The two are complementary.

**Why sweep k.** kNN depends on the same neighbourhood-size free parameter as LOF, with
no single correct value, so it is swept and summarised the same way: median rank across
the sweep, plus a stability count. The sweep uses the same K_VALUES bracket as LOF, so
the two are directly comparable rather than one method being scrutinised for
k-sensitivity while the other is taken on a single, unexamined value.

In [4]:
# Same sweep-and-summarise treatment as LOF, over the same K_VALUES bracket: score with
# knn_scores at each k, rank, then take the median rank across the sweep and count how
# often each burst lands in the top 10%.
knn_ranks = pd.DataFrame(index=scaled.index)
for k in K_VALUES:
    score_k = knn_scores(X, k=k)
    knn_ranks[k] = pd.Series(score_k, index=scaled.index).rank(ascending=False, method='first').astype(int)

results['knn_median_rank'] = knn_ranks.median(axis=1)
results['knn_rank'] = results['knn_median_rank'].rank(ascending=True, method='first').astype(int)
results['knn_stability'] = (knn_ranks <= top_n).sum(axis=1)

top = results.sort_values('knn_rank').head(10)
detail = knn_ranks.loc[top.index].copy()
detail.columns = [f'k={k}' for k in K_VALUES]
show = pd.concat([scaled.loc[top.index, ['tns_name', 'sub_num', 'is_repeater']], detail,
                  results.loc[top.index, ['knn_stability']]], axis=1)
print('Top 10 by kNN median rank, with rank at each k:')
print(show.to_string(index=False))
print(f'\nRepeaters in this top 10: {int(scaled.loc[top.index, "is_repeater"].sum())} of 10')
print(f'(top tier = top {top_n} rows = top 10%; stability is out of {len(K_VALUES)} sweeps)')

Top 10 by kNN median rank, with rank at each k:
    tns_name  sub_num  is_repeater  k=30  k=50  k=75  k=100  k=150  knn_stability
FRB20220407A        0        False     1     1     1      1      1              5
FRB20200723B        0        False     3     3     2      2      3              5
FRB20201126C        0        False     2     2     3      4      5              5
FRB20211108A        1        False     4     4     4      3      4              5
FRB20210909B        0        False    10     8     5      5      2              5
FRB20200705D        0        False     6     5     6      6      7              5
FRB20200122B        1         True     5     6     7      7      9              5
FRB20221219D        0         True     7     7     8      8      8              5
FRB20190116C        0        False     9    10     9     10     10              5
FRB20230826A        1         True     8     9    10     11     12              5

Repeaters in this top 10: 3 of 10
(top tier = top

## Method 4: Cluster-Based Local Outlier Factor (CBLOF)

**Principle: distance to the nearest large cluster.** The data is partitioned into
clusters, split into "large" and "small" groups by size. Each burst is scored by its
distance to the nearest large cluster, so a burst in a small cluster set apart from the
main mass scores high. This is the method in the set aimed at collective (micro-cluster)
anomalies: a small tight group of similar oddballs, which density methods read as normal
because the members are each other's neighbours.

In [5]:
# Implementation (hand-rolled KMeans-based CBLOF, He, Xu and Deng 2003) is documented in
# cblof_scores; see the principle paragraph above for the rationale.
d = cblof_scores(X)
results['cblof_score'] = d
results['cblof_rank'] = results['cblof_score'].rank(ascending=False, method='first').astype(int)

top10 = results.sort_values('cblof_rank').head(10)
print('CBLOF top 10:')
print(top10[['cblof_rank', 'tns_name', 'sub_num', 'cblof_score']].round({'cblof_score': 2}).to_string(index=False))

CBLOF top 10:
 cblof_rank     tns_name  sub_num  cblof_score
          1 FRB20220407A        0         7.75
          2 FRB20210909B        0         6.13
          3 FRB20190202A        0         6.12
          4 FRB20211108A        1         6.05
          5 FRB20201015B        0         5.94
          6 FRB20230911A        0         5.76
          7 FRB20200723B        0         5.74
          8 FRB20210427A        0         5.45
          9 FRB20190816B        0         5.42
         10 FRB20220222B        0         5.27


## Method 5: Random-subspace ensemble (feature bagging)

**Principle: anomalies hiding in a feature combination.** A burst can be unremarkable on
every single feature yet sit somewhere no normal burst does once two or more features are
viewed together (a broken correlation, an odd joint value). Full-dimensional distance and
single-feature methods miss these. Feature bagging (Lazarevic and Kumar 2005) runs a base
density detector on many random subsets of the features and aggregates the per-burst ranks,
so an anomaly that only shows up in some subspace is surfaced. This is the set's subspace
detector, and it replaces the dropped Mahalanobis as the correlation-aware method without
assuming the data is a single Gaussian blob.

In [6]:
# Feature-bagging ensemble (Lazarevic and Kumar 2005): mean-aggregated ranks across
# T_SUBSPACES random feature subsets, LOF base. Implementation is documented in
# subspace_scores; see the principle paragraph above for the rationale. Mean aggregation
# is the conservative choice for this overview; the max aggregation is compared against
# it directly in the candidate-validation notebook's stability diagnostics.
T_SUBSPACES = 50
s = subspace_scores(X, seed=42, agg='mean', n_subspaces=T_SUBSPACES, k=20)

results['subspace_score'] = s
results['subspace_rank'] = results['subspace_score'].rank(ascending=False, method='first').astype(int)

top10 = results.sort_values('subspace_rank').head(10)
print(f'Random-subspace ensemble ({T_SUBSPACES} subspaces, LOF base). Top 10:')
print(top10[['subspace_rank', 'tns_name', 'sub_num', 'subspace_score']].round({'subspace_score': 1}).to_string(index=False))

Random-subspace ensemble (50 subspaces, LOF base). Top 10:
 subspace_rank     tns_name  sub_num  subspace_score
             1 FRB20201119A        0          4095.5
             2 FRB20181119D        1          4093.6
             3 FRB20200124E        0          4090.1
             4 FRB20221030C        0          4081.0
             5 FRB20210222B        1          4079.8
             6 FRB20230422B        0          4075.4
             7 FRB20200427A        1          4073.7
             8 FRB20221219D        0          4065.7
             9 FRB20210829B        0          4061.8
            10 FRB20200321E        0          4056.1


## Method 6: Conditional anomaly detection (CAD), non-parametric, swept over k

**Principle: odd for its context.** Following the spirit of Song et al. (2007) but without
the Gaussian mixture that proved unstable here, the features split into observational
context (DM as a distance proxy, peak frequency as band position) and intrinsic morphology
(the rest). For each burst, its nearest neighbours in CONTEXT space are found, and it is
scored by how far its morphology sits from those context-peers. A burst whose shape is
unremarkable overall but unusual for bursts at its distance and band scores high. This is
the only method asking that conditional question, and it is fully non-parametric: no
component count, the only knob is the context-neighbourhood size, exactly LOF's k in a
different space.

**Why sweep k.** The same free-parameter problem as LOF and kNN applies here: no single
context-neighbourhood size is obviously correct, so it is swept and summarised the same
way, over the same K_VALUES bracket.

In [7]:
# Context/behaviour split is fixed regardless of k: DM and peak frequency define the
# observational context, the rest is intrinsic morphology.
context_cols = ['dm_fitb', 'peak_freq']
ctx_idx = [feature_cols.index(c) for c in context_cols]
beh_idx = [i for i in range(X.shape[1]) if i not in ctx_idx]

# Same sweep-and-summarise treatment as LOF and kNN, over the same K_VALUES bracket:
# score with cad_scores at each k, rank, then take the median rank across the sweep and
# count how often each burst lands in the top 10%.
cad_ranks = pd.DataFrame(index=scaled.index)
for k in K_VALUES:
    score_k = cad_scores(X, ctx_idx, beh_idx, k=k)
    cad_ranks[k] = pd.Series(score_k, index=scaled.index).rank(ascending=False, method='first').astype(int)

results['cad_median_rank'] = cad_ranks.median(axis=1)
results['cad_rank'] = results['cad_median_rank'].rank(ascending=True, method='first').astype(int)
results['cad_stability'] = (cad_ranks <= top_n).sum(axis=1)

top = results.sort_values('cad_rank').head(10)
detail = cad_ranks.loc[top.index].copy()
detail.columns = [f'k={k}' for k in K_VALUES]
show = pd.concat([scaled.loc[top.index, ['tns_name', 'sub_num', 'is_repeater']], detail,
                  results.loc[top.index, ['cad_stability']]], axis=1)
print('Top 10 by CAD median rank (context: DM, peak frequency), with rank at each k:')
print(show.to_string(index=False))
print(f'\nRepeaters in this top 10: {int(scaled.loc[top.index, "is_repeater"].sum())} of 10')
print(f'(top tier = top {top_n} rows = top 10%; stability is out of {len(K_VALUES)} sweeps)')

Top 10 by CAD median rank (context: DM, peak frequency), with rank at each k:
    tns_name  sub_num  is_repeater  k=30  k=50  k=75  k=100  k=150  cad_stability
FRB20210909B        0        False     1     1     1      1      1              5
FRB20220407A        0        False     2     2     2      2      2              5
FRB20190202A        0        False     3     3     3      3      3              5
FRB20211005A        0        False     4     4     6      5      8              5
FRB20230911A        0        False     6     5     4      6      7              5
FRB20210415C        0         True     5     6     8      7      9              5
FRB20220222B        0        False    16    14     7      4      4              5
FRB20200723B        0        False    10     8     5      9      6              5
FRB20201126C        0        False     9     7     9     10     11              5
FRB20201015B        0        False    13    11    10      8      5              5

Repeaters in this t

In [8]:
# Combine every method's per-burst output into one table and write it to disk. Persist
# each method's RAW quantity and fix the direction downstream, so nothing is silently
# transformed here.
#
# Direction of each column:
#   lof_median_rank   LOWER  = more anomalous  (rank across the k-sweep, not a score)
#   eif_score         higher = more anomalous
#   knn_median_rank   LOWER  = more anomalous  (rank across the k-sweep, not a score)
#   cblof_score       higher = more anomalous
#   subspace_score    higher = more anomalous  (mean rank across random subspaces)
#   cad_median_rank   LOWER  = more anomalous  (rank across the k-sweep, not a score)
#
# repeater_name (blank/NaN for one-offs) and catalog1_flag are carried as metadata only;
# the scoring above is label-blind. 'results' was built in row order and never reordered,
# so scaled's columns line up with it positionally.
score_cols = ['lof_median_rank', 'eif_score', 'knn_median_rank', 'cblof_score',
              'subspace_score', 'cad_median_rank']
scores = results[['tns_name', 'sub_num', 'is_repeater', *score_cols]].copy()
scores['repeater_name'] = scaled['repeater_name'].values
scores['catalog1_flag'] = scaled['catalog1_flag'].values
scores = scores[['tns_name', 'sub_num', 'repeater_name', 'catalog1_flag', 'is_repeater',
                 *score_cols]]

out_path = PROJECT / 'data' / 'processed' / 'phase_1' / 'catalog2_method_scores.csv'
scores.to_csv(out_path, index=False)

print(f'Wrote {len(scores)} rows to {out_path.relative_to(PROJECT)}')
print(f'Columns: {list(scores.columns)}')

Wrote 4109 rows to data/processed/phase_1/catalog2_method_scores.csv
Columns: ['tns_name', 'sub_num', 'repeater_name', 'catalog1_flag', 'is_repeater', 'lof_median_rank', 'eif_score', 'knn_median_rank', 'cblof_score', 'subspace_score', 'cad_median_rank']
